In [ ]:
#!/usr/bin/env python3
"""
volume_to_weight_train.py

Pipeline:
 - For each grayscale image:
   * Otsu threshold -> binary mask
   * Compute bounding box of mask -> (minx, miny, maxx, maxy)
   * Split vertical bbox into N horizontal slices
   * Estimate cross-sectional area at slice boundaries using row widths:
       A(row) = (pi/4) * (width_in_pixels)^2   [assume circular cross-section]
   * For internal slices: treat as frustum
       V = (h/3) * (A1 + A2 + sqrt(A1*A2))
     For first/last slice: treat as cone:
       V = (h/3) * A_base
   * Sum volumes -> estimated_volume
 - Build polynomial features from estimated_volume (v, v^2, ... v^D)
 - Train linear model y = w^T x + b using PyTorch with elastic-net penalty
 - Hyperparameters: learning rate (lr), alpha (overall reg strength), l1_ratio (L1 vs L2)
 - LR updated by ReduceLROnPlateau; alpha decayed when validation plateaus
 - Save trained model and scaler/metadata
"""

import os
import argparse
from typing import Tuple, List, Dict
import math
import pickle
import numpy as np
from PIL import Image
import csv
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset

# ----------------------------
# Utilities: Otsu thresholding
# ----------------------------
def otsu_threshold_from_array(gray: np.ndarray) -> int:
    """Compute Otsu threshold for 2D uint8 numpy array (0..255)."""
    hist, _ = np.histogram(gray.ravel(), bins=256, range=(0, 256))
    total = hist.sum()
    if total == 0:
        return 0
    prob = hist.astype(np.float64) / total
    omega = np.cumsum(prob)
    mu = np.cumsum(prob * np.arange(256))
    mu_total = mu[-1]
    # avoid division by zero; add tiny eps in denom
    denom = omega * (1.0 - omega) + 1e-12
    sigma_b2 = (mu_total * omega - mu) ** 2 / denom
    sigma_b2[omega == 0] = 0
    sigma_b2[omega == 1] = 0
    t = int(np.argmax(sigma_b2))
    return t

# ----------------------------
# Mask / bbox / volume utils
# ----------------------------
def image_to_mask(img_path: str, otsu_blur: bool = True) -> np.ndarray:
    """
    Read grayscale image (PIL), convert to numpy uint8, threshold with Otsu => binary mask (0/1)
    """
    img = Image.open(img_path).convert("L")
    arr = np.array(img)
    # optional smoothing (small gaussian blur) to reduce noise; using simple uniform filter could also be used
    # we'll omit heavy dependencies; small median filter via numpy: we skip for simplicity
    t = otsu_threshold_from_array(arr)
    mask = (arr > t).astype(np.uint8)
    return mask

def bbox_from_mask(mask: np.ndarray) -> Tuple[int,int,int,int]:
    """
    Return bounding box (minx, miny, maxx, maxy) in image coordinates for pixels where mask==1.
    If no pixels, return (0,0,0,0)
    """
    ys, xs = np.where(mask > 0)
    if len(xs) == 0:
        return (0,0,0,0)
    minx, maxx = int(xs.min()), int(xs.max())
    miny, maxy = int(ys.min()), int(ys.max())
    return (minx, miny, maxx, maxy)

def avg_row_width(mask: np.ndarray, y: int, minx: int, maxx: int, half_window: int = 1) -> float:
    """
    Return average white-pixels count across rows y-half_window .. y+half_window clipped to bbox.
    This estimates width (in pixels) of cross-section at row y.
    """
    H, W = mask.shape
    ys = list(range(max(0, y-half_window), min(H, y+half_window+1)))
    if not ys:
        return 0.0
    widths = []
    for yy in ys:
        widths.append(int(np.sum(mask[yy, minx:maxx+1] > 0)))
    return float(np.mean(widths))

def estimate_volume_from_mask(mask: np.ndarray, N_slices: int, bbox: Tuple[int,int,int,int], boundary_window: int = 1) -> Tuple[float, List[float]]:
    """
    Given binary mask and bbox, split into N_slices and estimate volume in pixel-units^3.
    Returns (total_volume, list_of_slice_volumes).
    Uses width-based cross-sectional area: A = (pi/4) * (width_pixels)^2 (assume circular cross-sections)
    """
    minx, miny, maxx, maxy = bbox
    if minx == maxx and miny == maxy:
        # degenerate or empty
        return 0.0, [0.0] * N_slices

    # height of bbox in pixels (vertical length)
    L = maxy - miny + 1
    # slice height in pixels (can be non-integer; we'll treat as integer slices)
    slice_h = float(L) / float(N_slices)

    # Precompute area function (at arbitrary y we approximate cross-sectional area)
    def area_at_row(y_float: float) -> float:
        # round to nearest int row index
        y = int(round(y_float))
        # ensure within bbox
        y = max(miny, min(maxy, y))
        width = avg_row_width(mask, y, minx, maxx, half_window=boundary_window)
        # compute area assuming circular cross-section: A = pi/4 * width^2
        A = (math.pi / 4.0) * (width ** 2)
        return A

    slice_volumes = []
    total_vol = 0.0
    for i in range(N_slices):
        # top and bottom of slice in image coords (float)
        top_f = miny + i * slice_h
        bottom_f = miny + (i + 1) * slice_h
        # height of this slice
        h = bottom_f - top_f
        # For frustum between top and bottom we need A_top and A_bottom
        A_top = area_at_row(top_f)
        A_bottom = area_at_row(bottom_f)
        if i == 0:
            # head slice => elliptical cone (apex at top), base area = A_bottom (we choose base as bottom area)
            V = (h / 3.0) * A_bottom
        elif i == N_slices - 1:
            # tail slice => cone with base area = A_top (apex at bottom)
            V = (h / 3.0) * A_top
        else:
            # frustum
            V = (h / 3.0) * (A_top + A_bottom + math.sqrt(max(0.0, A_top * A_bottom)))
        slice_volumes.append(V)
        total_vol += V

    # If first or last slices weren't appended due to logic, ensure length N_slices
    # (we appended for middle slices only; fix ordering)
    # Actually our loop only appended for middle slices — adjust: rebuild slice_volumes properly
    # We'll reconstruct correct list:
    slice_volumes_full = []
    total_vol = 0.0
    for i in range(N_slices):
        top_f = miny + i * slice_h
        bottom_f = miny + (i + 1) * slice_h
        h = bottom_f - top_f
        A_top = area_at_row(top_f)
        A_bottom = area_at_row(bottom_f)
        if i == 0:
            V = (h / 3.0) * A_bottom
        elif i == N_slices - 1:
            V = (h / 3.0) * A_top
        else:
            V = (h / 3.0) * (A_top + A_bottom + math.sqrt(max(0.0, A_top * A_bottom)))
        slice_volumes_full.append(V)
        total_vol += V

    return total_vol, slice_volumes_full

# ----------------------------
# Data loading & preprocessing
# ----------------------------
def load_csv_pairs(csv_file: str, img_root: str = "") -> List[Tuple[str, float]]:
    """
    CSV expected format: image_path,weight
    image_path may be relative to img_root or absolute.
    """
    pairs = []
    with open(csv_file, "r", newline='') as f:
        reader = csv.reader(f)
        header = next(reader, None)
        # if header exists and contains 'image' or 'path' or 'weight', skip accordingly
        # try to detect header line if non-numeric weight in second column
        # we'll assume header present if any entry non-parsable to float in second col
        # To be robust: if header appears to be text, skip it already.
        # We'll detect by trying to parse first data row's second column.
        # If header seems to be present we have already consumed it.
        # If header is actual data, we already have it as first row and will process below.
        # For simplicity, if header contains 'image' or 'weight' skip earlier consumed header.
        # Try to interpret header:
        if header and (header[0].lower().find('image') != -1 or header[1].lower().find('weight') != -1 or header[0].lower().find('path')!=-1):
            # treat header consumed, proceed
            pass
        else:
            # header might be actual first data row; rewind
            if header is not None:
                try:
                    # try parse weight
                    _ = float(header[1])
                    # treat as data row
                    img_path = header[0]
                    weight = float(header[1])
                    full_path = os.path.join(img_root, img_path) if img_root and not os.path.isabs(img_path) else img_path
                    pairs.append((full_path, weight))
                except Exception:
                    # header wasn't numeric second column, ignore
                    pass

        for row in reader:
            if len(row) < 2:
                continue
            img_path = row[0]
            try:
                weight = float(row[1])
            except:
                continue
            full_path = os.path.join(img_root, img_path) if img_root and not os.path.isabs(img_path) else img_path
            pairs.append((full_path, weight))
    return pairs

# ----------------------------
# Polynomial features & scaler
# ----------------------------
class SimpleScaler:
    """Simple mean-std scaler for 2D array (n_samples x n_features)."""
    def __init__(self):
        self.mean_ = None
        self.std_ = None

    def fit(self, X: np.ndarray):
        self.mean_ = np.mean(X, axis=0, keepdims=True)
        self.std_ = np.std(X, axis=0, keepdims=True)
        # avoid zero
        self.std_[self.std_ == 0.0] = 1.0

    def transform(self, X: np.ndarray) -> np.ndarray:
        return (X - self.mean_) / self.std_

    def fit_transform(self, X: np.ndarray) -> np.ndarray:
        self.fit(X)
        return self.transform(X)

# polynomial features for scalar volume v: [v, v^2, ..., v^D]
def poly_features_from_volume(vols: np.ndarray, degree: int) -> np.ndarray:
    n = vols.shape[0]
    feats = np.zeros((n, degree), dtype=np.float64)
    for d in range(1, degree+1):
        feats[:, d-1] = vols[:, 0] ** d
    return feats

# ----------------------------
# PyTorch model (simple linear)
# ----------------------------
class PolyRegModel(nn.Module):
    def __init__(self, in_features: int):
        super().__init__()
        self.lin = nn.Linear(in_features, 1)  # bias included

    def forward(self, x):
        # x shape (batch, in_features)
        return self.lin(x).squeeze(1)  # (batch,)

# ----------------------------
# Training / Validation loop
# ----------------------------
def train_model(volumes_train: np.ndarray, weights_train: np.ndarray,
                volumes_val: np.ndarray, weights_val: np.ndarray,
                degree: int,
                lr: float = 1e-3,
                alpha: float = 1e-3,
                l1_ratio: float = 0.5,
                batch_size: int = 8,
                epochs: int = 50,
                patience: int = 5,
                alpha_decay: float = 0.5,
                device: str = 'cpu'):
    """Train polynomial regression with elastic-net regularization using PyTorch.
    Returns trained model, scaler, training history, best_alpha used.
    """
    # Build polynomial features
    X_train_raw = poly_features_from_volume(volumes_train.reshape(-1,1), degree)  # shape (n, degree)
    X_val_raw = poly_features_from_volume(volumes_val.reshape(-1,1), degree)

    # Fit scaler on training features
    scaler = SimpleScaler()
    X_train = scaler.fit_transform(X_train_raw)
    X_val = scaler.transform(X_val_raw)

    y_train = weights_train.astype(np.float32)
    y_val = weights_val.astype(np.float32)

    # Convert to tensors
    train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
    val_ds = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.float32))

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    device = torch.device(device)
    model = PolyRegModel(in_features=degree).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    # LR scheduler on plateau
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=patience, verbose=True)

    best_val_loss = float('inf')
    best_state = None
    best_alpha = alpha
    no_improve = 0

    history = {'train_loss': [], 'val_loss': [], 'alpha': []}

    for epoch in range(1, epochs+1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad()
            preds = model(xb)
            mse = torch.mean((preds - yb) ** 2)
            # Elastic-net penalty
            l1_pen = 0.0
            l2_pen = 0.0
            for param in model.parameters():
                l1_pen = l1_pen + torch.sum(torch.abs(param))
                l2_pen = l2_pen + torch.sum(param ** 2)
            # combine: alpha * (l1_ratio * L1 + (1-l1_ratio) * 0.5 * L2)
            reg = alpha * (l1_ratio * l1_pen + (1.0 - l1_ratio) * 0.5 * l2_pen)
            loss = mse + reg
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())
        avg_train_loss = float(np.mean(train_losses)) if train_losses else 0.0

        # Validation
        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                mse = torch.mean((preds - yb) ** 2)
                # regularization applied for validation loss evaluation as well to reflect overall objective
                l1_pen = 0.0
                l2_pen = 0.0
                for param in model.parameters():
                    l1_pen = l1_pen + torch.sum(torch.abs(param))
                    l2_pen = l2_pen + torch.sum(param ** 2)
                reg = alpha * (l1_ratio * l1_pen + (1.0 - l1_ratio) * 0.5 * l2_pen)
                loss = mse + reg
                val_losses.append(loss.item())
        avg_val_loss = float(np.mean(val_losses)) if val_losses else 0.0

        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['alpha'].append(alpha)

        # Scheduler step (ReduceLROnPlateau)
        scheduler.step(avg_val_loss)

        if avg_val_loss < best_val_loss - 1e-8:
            best_val_loss = avg_val_loss
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            best_alpha = alpha
            no_improve = 0
            print(f"[Epoch {epoch}] Improved val loss: {avg_val_loss:.6f}")
        else:
            no_improve += 1
            print(f"[Epoch {epoch}] No improvement ({no_improve}/{patience}). Val loss: {avg_val_loss:.6f}")

        # Adjust alpha hyperparameter if no improvement for patience epochs
        if no_improve >= patience:
            old_alpha = alpha
            alpha = max(1e-12, alpha * alpha_decay)  # reduce regularization strength
            print(f"  -> Decayed alpha {old_alpha} -> {alpha}")
            no_improve = 0  # reset

        print(f"Epoch {epoch}/{epochs}: train_loss={avg_train_loss:.6f}, val_loss={avg_val_loss:.6f}, alpha={alpha}, lr={optimizer.param_groups[0]['lr']}")

    # load best state
    if best_state is not None:
        model.load_state_dict(best_state)

    return model, scaler, history, best_alpha

# ----------------------------
# Full pipeline: precompute volumes, split, train, save
# ----------------------------
def run_pipeline(csv_file: str, img_root: str, out_model_path: str,
                 N_slices: int = 20, degree: int = 3, lr: float = 1e-3,
                 alpha: float = 1e-3, l1_ratio: float = 0.5,
                 epochs: int = 50, batch_size: int = 8,
                 val_split: float = 0.2, random_seed: int = 42,
                 device: str = 'cpu'):
    pairs = load_csv_pairs(csv_file, img_root)
    if len(pairs) == 0:
        raise ValueError("No (image,weight) pairs loaded. Check CSV file.")

    # Precompute volumes and bounding boxes
    volumes = []
    bboxes = []
    img_paths = []
    weights = []
    print("Precomputing volumes for images (this will load each image once)...")
    for img_path, w in tqdm(pairs):
        try:
            mask = image_to_mask(img_path)
        except Exception as e:
            print(f"Warning: failed to load/process image {img_path} -> skipping. Error: {e}")
            continue
        bbox = bbox_from_mask(mask)
        vol, slice_vols = estimate_volume_from_mask(mask, N_slices, bbox)
        volumes.append(vol)
        bboxes.append(bbox)
        img_paths.append(img_path)
        weights.append(w)

    volumes = np.array(volumes, dtype=np.float64).reshape(-1, 1)
    weights = np.array(weights, dtype=np.float64).reshape(-1, 1)

    # Train/val split
    n = volumes.shape[0]
    rng = np.random.RandomState(random_seed)
    indices = np.arange(n)
    rng.shuffle(indices)
    val_n = max(1, int(math.ceil(val_split * n)))
    val_idx = indices[:val_n]
    train_idx = indices[val_n:]

    vols_train = volumes[train_idx].reshape(-1)
    w_train = weights[train_idx].reshape(-1)
    vols_val = volumes[val_idx].reshape(-1)
    w_val = weights[val_idx].reshape(-1)

    print(f"Samples: total={n}, train={len(train_idx)}, val={len(val_idx)}")

    model, scaler, history, best_alpha = train_model(
        vols_train, w_train, vols_val, w_val,
        degree=degree, lr=lr, alpha=alpha, l1_ratio=l1_ratio,
        batch_size=batch_size, epochs=epochs, device=device
    )

    # Save model + scaler + metadata
    meta = {
        'degree': degree,
        'scaler_mean': scaler.mean_,
        'scaler_std': scaler.std_,
        'best_alpha': best_alpha,
        'l1_ratio': l1_ratio
    }
    save_dict = {'model_state': model.state_dict(), 'meta': meta}
    torch.save(save_dict, out_model_path)
    print(f"Saved model to {out_model_path}")

    # Also save bbox info and computed volumes (so you can inspect)
    cache = {
        'img_paths': img_paths,
        'bboxes': bboxes,
        'volumes': volumes.reshape(-1).tolist(),
        'weights': weights.reshape(-1).tolist()
    }
    cache_path = out_model_path + ".cache.pkl"
    with open(cache_path, 'wb') as f:
        pickle.dump(cache, f)
    print(f"Saved cache (volumes/bboxes) to {cache_path}")

    return model, scaler, meta, history

# ----------------------------
# CLI
# ----------------------------
def parse_args():
    p = argparse.ArgumentParser()
    p.add_argument('--csv', required=True, help='CSV file with image_path,weight')
    p.add_argument('--img_root', default='', help='root dir to prefix to image_path in csv (optional)')
    p.add_argument('--out_model', required=True, help='Path to save trained model (e.g., model.pth)')
    p.add_argument('--N_slices', type=int, default=20)
    p.add_argument('--degree', type=int, default=3)
    p.add_argument('--lr', type=float, default=1e-3)
    p.add_argument('--alpha', type=float, default=1e-3)
    p.add_argument('--l1_ratio', type=float, default=0.5)
    p.add_argument('--epochs', type=int, default=50)
    p.add_argument('--batch_size', type=int, default=8)
    p.add_argument('--val_split', type=float, default=0.2)
    p.add_argument('--device', type=str, default='cpu')
    p.add_argument('--seed', type=int, default=42)
    return p.parse_args()

if __name__ == "__main__":
    args = parse_args()
    run_pipeline(
        csv_file=args.csv,
        img_root=args.img_root,
        out_model_path=args.out_model,
        N_slices=args.N_slices,
        degree=args.degree,
        lr=args.lr,
        alpha=args.alpha,
        l1_ratio=args.l1_ratio,
        epochs=args.epochs,
        batch_size=args.batch_size,
        val_split=args.val_split,
        random_seed=args.seed,
        device=args.device
    )